In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [ ]:
def quantum_rng(num_bits):
    """Generates a list of random bits using a quantum circuit."""
    from qiskit import QuantumCircuit
    from qiskit.providers.basic_provider import BasicSimulator
    from qiskit import transpile
    
    rng_circuits = []
    for _ in range(num_bits):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        rng_circuits.append(qc)
        
    simulator = BasicSimulator()
    transpiled_circuits = transpile(rng_circuits, simulator)
    job = simulator.run(transpiled_circuits, shots=1)
    result = job.result()
    
    bits = []
    for i in range(num_bits):
        counts = result.get_counts(i)
        bit = list(counts.keys())[0]
        bits.append(int(bit))
    return bits

N = 50 # Number of qubits for the protocol
print(f"Quantum RNG test (5 bits): {quantum_rng(5)}")

In [ ]:
# ==========================================
# ALICE'S PART
# ==========================================

print("--- ALICE ---")
alice_bits = quantum_rng(N)
alice_bases = quantum_rng(N)

print(f"Alice's bits:  {alice_bits[:10]}...")
print(f"Alice's bases: {alice_bases[:10]}... (0=Z, 1=X)")

qubits = []
for i in range(N):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    qubits.append(qc)

print(f"Alice prepared {len(qubits)} qubits and sent them out.")

In [ ]:
# ==========================================
# EVE'S PART (THE ATTACKER)
# ==========================================

print("\n--- EVE (ATTACKER) ---")
print("Eve intercepts the qubits!")

eve_bases = quantum_rng(N)
eve_results = []
simulator = BasicSimulator()

for i in range(N):
    # 1. Eve intercepts the qubit
    qc = qubits[i] 
    
    # 2. Eve measures the qubit in her randomly chosen basis
    if eve_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    
    transpiled_qc = transpile(qc, simulator)
    job = simulator.run(transpiled_qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    bit = int(list(counts.keys())[0])
    eve_results.append(bit)
    
    # 3. Eve prepares a new qubit to send to Bob based on her measurement result and basis
    new_qc = QuantumCircuit(1, 1)
    if bit == 1:
        new_qc.x(0)
    if eve_bases[i] == 1:
        new_qc.h(0)
        
    # Eve replaces Alice's qubit with her newly prepared one
    qubits[i] = new_qc

print(f"Eve's bases:   {eve_bases[:10]}...")
print(f"Eve's results: {eve_results[:10]}...")

In [ ]:
# ==========================================
# BOB'S PART
# ==========================================

print("\n--- BOB ---")
bob_bases = quantum_rng(N)
print(f"Bob's bases:   {bob_bases[:10]}...")

bob_results = []

for i in range(N):
    # Bob receives the (potentially tampered) qubits
    qc = qubits[i]
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    
    transpiled_qc = transpile(qc, simulator)
    job = simulator.run(transpiled_qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    bit = int(list(counts.keys())[0])
    bob_results.append(bit)

print(f"Bob's results: {bob_results[:10]}...")

In [ ]:
# ==========================================
# PUBLIC DISCUSSION & SIFTING
# ==========================================

print("\n--- PUBLIC DISCUSSION ---")
sifted_alice = []
sifted_bob = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Sifted Key length: {len(sifted_alice)}")
print(f"Sifted Key (Alice): {sifted_alice[:10]}...")
print(f"Sifted Key (Bob):   {sifted_bob[:10]}...")

In [ ]:
# ==========================================
# ERROR CHECKING
# ==========================================

print("\n--- ERROR CHECKING ---")
num_check = len(sifted_alice) // 2
check_alice = sifted_alice[:num_check]
check_bob = sifted_bob[:num_check]

errors = 0
for i in range(num_check):
    if check_alice[i] != check_bob[i]:
        errors += 1
        
error_rate = errors / num_check if num_check > 0 else 0
print(f"Bits checked: {num_check}")
print(f"Errors found: {errors}")
print(f"Error Rate: {error_rate*100:.2f}%")

if error_rate > 0.1:
    print("\n🚨 ATTACK DETECTED! The error rate exceeds the threshold. Protocol aborted.")
else:
    print("\n✅ No attack detected. Proceeding to use the final key.")
    final_key = sifted_alice[num_check:]
    print(f"Final Secret Key: {final_key}")